# Module 7: Engine Mechanics and Saturation

In Module 6 you added a small draft model and measured whether the accepted tokens paid for the extra work. Now zoom out to the serving engine. vLLM is fast because it manages the KV cache, uses efficient attention kernels, and continuously batches requests. This module drives rising concurrency against the deployment you measured, with FP8 and the current draft-path setting, until the useful capacity knee appears.


## Learning objectives
- Explain why attention, KV cache, and batching dominate serving behavior
- Describe PagedAttention and continuous batching
- Run a concurrency sweep against your vLLM endpoint
- Identify the saturation knee from throughput and latency
- Connect queueing, KV pressure, and preemptions to the next tuning step


## Prerequisites
- Finished Module 6
- A live FP8 vLLM endpoint from Module 6, with whatever speculative setting you decided to keep for measurement
- About 20 minutes


References: [Anatomy of vLLM](https://blog.vllm.ai/2025/09/05/anatomy-of-vllm.html) &middot; [vLLM production metrics](https://docs.vllm.ai/en/latest/design/metrics/) &middot; [Continuous batching](https://www.anyscale.com/blog/continuous-batching-llm-inference)


## Engine design basics

A single request rarely fills the GPU during decode. Continuous batching lets vLLM add and remove requests as tokens are generated, so one weight read can serve many requests at the same time. That raises throughput until something else becomes the limit: scheduler caps, KV cache, waiting requests, or latency.

![Incoming requests flow through a queue into a running batch while the saturation curve rises to a knee and then latency grows](images/07_inference_engine_saturation_architecture.svg)


## 1. Setup

Install the small client dependencies and resolve your endpoint. The load helper in `common/loadtest.py` sends concurrent streaming requests and returns one row per concurrency level.


In [ ]:

%pip install -q "openai>=1.40" "requests>=2.31"


In [ ]:

# Imports and settings for the live sweep.
import os, sys
from pathlib import Path

if Path("../manifests/vllm.yaml").exists():
    REPO_ROOT = Path("..")
else:
    REPO_ROOT = Path(".")

sys.path.insert(0, str(REPO_ROOT.resolve()))

from common.config import print_settings
from common import loadtest

settings = print_settings()
TARGET_MODEL = "RedHatAI/Qwen3-4B-FP8-dynamic"
print("target:", TARGET_MODEL)

import requests
root = settings.vllm_host.rstrip("/").removesuffix("/v1")

def served_models():
    data = requests.get(
        f"{root}/v1/models",
        headers={"Authorization": f"Bearer {settings.api_key}"},
        timeout=20,
    ).json()
    return [item["id"] for item in data.get("data", [])]

models = served_models()
print("served models:", models)
assert TARGET_MODEL in models, f"Expected {TARGET_MODEL}; got {models}"


**What you should see:** the same endpoint you configured in Module 6, the FP8 target model id, and that model in the served-model list. If the endpoint is not reachable, go back to Module 0's pod and `/v1/models` checks.


## 2. Attention and the KV cache

Attention is expensive because every new token attends over prior context. The KV cache keeps prior keys and values so the server does not recompute the full prompt. PagedAttention is vLLM's memory manager for that cache: it lets many uneven requests share GPU memory without needing one huge contiguous block per request.


## 3. Continuous batching

Continuous batching is the serving trick you can see from the outside. At low concurrency, throughput rises as more requests share the GPU. At the knee, throughput stops climbing while latency and waiting start climbing. That knee is your useful operating boundary for this model, GPU, speculative setting, and scheduler configuration.


In [ ]:

# Requires a live vLLM endpoint. Run a small saturation walk.
# Use the served target model explicitly; MODEL_NAME may still be stale after manifest edits.
levels = [1, 8, 32, 64, 128]
rows = loadtest.sweep(levels, input_tokens=256, output_tokens=128, model=TARGET_MODEL)
rows


**What you should see:** throughput rises from low concurrency, then flattens. TTFT usually grows near or after the knee. If all levels fail, check `VLLM_HOST`, the model id, and whether the vLLM pod is Ready.


## 4. Read the knee

Print the rows and find the first level where throughput barely moves but latency rises. That is not a failure. It is the capacity boundary you need before Module 8 can tune anything responsibly.


In [ ]:

# Inspect the raw rows returned by the helper.
for row in rows:
    print(row)


**What you should see:** one dictionary per concurrency level. Keep the highest useful throughput and the TTFT p95 near the knee; those become your Module 8 baseline.


## 5. What the metrics mean

- Throughput rising and TTFT stable: the GPU had idle room and batching is helping.
- Throughput flat and TTFT rising: you passed the useful knee.
- Waiting requests rising: scheduler or batch limits are probably binding.
- KV near full or preemptions rising: cache pressure is probably binding.

The exact knee is less important than the method: sweep, observe, name the first bottleneck, then change one thing.


## Things to know

- **Saturation is information.** You are finding the boundary, not breaking the system.
- **Batching shares weight reads.** That is why decode gets better under concurrency even though one request does not get much faster.
- **Long prompts move the knee.** More prefill and more KV cache per request change the limiting resource.
- **The next module needs this baseline.** Do not tune before you can describe the current knee.


## Try it yourself

**Change prompt length.** Run the sweep again with `input_tokens=1024`. Watch whether the knee moves earlier and whether TTFT grows faster.

**Change output length.** Run with `output_tokens=32`, then `output_tokens=256`. Short outputs emphasize request overhead; long outputs emphasize decode.


## Summary

- vLLM uses cache management and continuous batching to keep the GPU busy.
- Throughput rises with concurrency until another resource becomes the limit.
- The saturation knee is visible in throughput, TTFT, queueing, KV pressure, and preemptions.
- You now have the baseline that Module 8 tunes against.


## Next

**Module 8: Tune and Evaluate.** You found the bottleneck. Next you edit the same shared manifest, change one vLLM flag at a time, redeploy, and decide whether to keep or reject the new operating point.
